### Automated Stacking Ensemble Regression Pipeline with Cross-Validation and Multi-Seed Evaluation

In [1]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, StackingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

BASE_INPUT_DIR = ""

# FUNCTION TO RUN PIPELINE PER FOLDER
def run_regression_pipeline(folder_path, target_prefix):
    dataset_name = os.path.basename(folder_path)
    print(f"\n{'='*80}\n Processing Folder: {dataset_name}\n{'='*80}")

    try:
        # -------------------------------------------------------------
        # Load Data
        # -------------------------------------------------------------
        X_train_scaled = pd.read_csv(os.path.join(folder_path, f"Solubility_AqSolDB_train_fp_features_scaled.csv"))
        y_train = pd.read_csv(os.path.join(folder_path, f"Solubility_AqSolDB_train_fp_features_raw.csv")).Label

        X_val_scaled = pd.read_csv(os.path.join(folder_path, f"Solubility_AqSolDB_valid_fp_features_scaled.csv"))
        y_val = pd.read_csv(os.path.join(folder_path, f"Solubility_AqSolDB_valid_fp_features_raw.csv")).Label

        X_test_scaled = pd.read_csv(os.path.join(folder_path, f"Solubility_AqSolDB_test_fp_features_scaled.csv"))
        y_test = pd.read_csv(os.path.join(folder_path, f"Solubility_AqSolDB_test_fp_features_raw.csv")).Label

        # -------------------------------------------------------------
        # Save input shapes
        # -------------------------------------------------------------
        shapes_df = pd.DataFrame({
            'Split': ['Train', 'Valid', 'Test'],
            'X_shape': [str(X_train_scaled.shape), str(X_val_scaled.shape), str(X_test_scaled.shape)],
            'y_shape': [str(y_train.shape), str(y_val.shape), str(y_test.shape)]
        })
        save_path_output = os.path.join(folder_path, "Outputs")
        os.makedirs(save_path_output, exist_ok=True)
        shapes_df.to_csv(os.path.join(save_path_output, "data_shapes.csv"), index=False)

        # -------------------------------------------------------------
        # Prepare data
        # -------------------------------------------------------------
        X_train_fp_clean = X_train_scaled
        X_valid_fp_clean = X_val_scaled
        X_test_fp_clean  = X_test_scaled
        y_train_fp_clean = y_train
        y_valid_fp_clean = y_val
        y_test_fp_clean  = y_test

        all_results = []

        # -------------------------------------------------------------
        # Run multiple random seeds
        # -------------------------------------------------------------
        for seed in [1, 2, 3, 4, 5]:
            print(f"\n=== Training Run with seed={seed} ===")

            base_models = [
                ('et', ExtraTreesRegressor(random_state=seed, n_jobs=-1)),
                ('rf', RandomForestRegressor(random_state=seed, n_jobs=-1)),
                # ('lgbm', LGBMRegressor(device='cpu', random_state=seed, force_col_wise=True, verbose=-1)),
                ('lgbm', LGBMRegressor(device_type='gpu', gpu_platform_id=0, gpu_device_id=0, random_state=seed, force_col_wise=True, verbose=-1)),
                ('xgb', XGBRegressor(tree_method='hist', device='cuda', random_state=seed, eval_metric='rmse'))
            ]

            meta_learner = Ridge(random_state=42)
            stacking_ensemble = StackingRegressor(
                estimators=base_models,
                final_estimator=meta_learner,
                n_jobs=-1,
                cv=10
            )

            # Combine train + val
            X_train_full = pd.concat([X_train_fp_clean, X_valid_fp_clean], axis=0, ignore_index=True)
            y_train_full = np.concatenate([y_train_fp_clean, y_valid_fp_clean])

            stacking_ensemble.fit(X_train_full, y_train_full)

            # Predict
            y_pred = stacking_ensemble.predict(X_test_fp_clean)

            # Metrics
            mae = mean_absolute_error(y_test_fp_clean, y_pred)
            rmse = mean_squared_error(y_test_fp_clean, y_pred)
            r2 = r2_score(y_test_fp_clean, y_pred)
            spearman_corr, _ = spearmanr(y_test_fp_clean, y_pred)

            # Save predictions
            pred_df = pd.DataFrame({'y_true': y_test_fp_clean, 'y_pred': y_pred})
            pred_file = os.path.join(save_path_output, f"Regression_predictions_seed{seed}.csv")
            pred_df.to_csv(pred_file, index=False)

            # Save model
            model_file = os.path.join(save_path_output, f"Regression_model_seed{seed}.pkl")
            joblib.dump(stacking_ensemble, model_file)

            # Store results
            all_results.append({
                'Seed': seed,
                'MAE': mae,
                'RMSE': rmse,
                'R2': r2,
                'Spearman': spearman_corr,
                'Model_File': model_file,
                'Prediction_File': pred_file
            })

            print(f"Seed {seed} -> MAE {mae:.4f}, RMSE {rmse:.4f}, R2 {r2:.4f}, Spearman {spearman_corr:.4f}")

        # -------------------------------------------------------------
        # Save aggregated results
        # -------------------------------------------------------------
        results_df = pd.DataFrame(all_results)
        results_df.to_csv(os.path.join(save_path_output, "Regression_all_results.csv"), index=False)

        summary = results_df[['MAE','RMSE','R2','Spearman']].agg(['mean','std']).T
        summary['Mean ± Std'] = summary.apply(lambda x: f"{x['mean']:.4f} ± {x['std']:.4f}", axis=1)
        summary.to_csv(os.path.join(save_path_output, "Regression_summary.csv"))

        print("\n=== FINAL SUMMARY ===")
        print(summary[['Mean ± Std']])

    except Exception as e:
        print(f"❌ Error processing {dataset_name}: {e}")


# RUN FOR ALL SUBFOLDERS
target_prefix = "DatasetPrefix"  # e.g., "CYP3A4", "Solubility"
for subfolder in sorted(os.listdir(BASE_INPUT_DIR)):
    folder_path = os.path.join(BASE_INPUT_DIR, subfolder)
    if os.path.isdir(folder_path):
        run_regression_pipeline(folder_path, target_prefix)



 Processing Folder: 05_14_TDC_morgan_avalon_erg_tfidf_embed64_variance

=== Training Run with seed=1 ===


C:\Users\omidm\anaconda3\envs\myenv310-2\lib\site-packages\xgboost\core.py:774: UserWarning: [16:58:15] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Seed 1 -> MAE 0.6798, RMSE 0.9486, R2 0.8252, Spearman 0.8988

=== Training Run with seed=2 ===
Seed 2 -> MAE 0.6810, RMSE 0.9498, R2 0.8250, Spearman 0.8984

=== Training Run with seed=3 ===
Seed 3 -> MAE 0.6816, RMSE 0.9500, R2 0.8249, Spearman 0.8986

=== Training Run with seed=4 ===
Seed 4 -> MAE 0.6829, RMSE 0.9517, R2 0.8246, Spearman 0.8981

=== Training Run with seed=5 ===
Seed 5 -> MAE 0.6828, RMSE 0.9493, R2 0.8251, Spearman 0.8985

=== FINAL SUMMARY ===
               Mean ± Std
MAE       0.6816 ± 0.0013
RMSE      0.9499 ± 0.0012
R2        0.8250 ± 0.0002
Spearman  0.8985 ± 0.0002

 Processing Folder: 06_01_TDC_morgan_avalon_erg_selfies_rdkit_maccs_variance

=== Training Run with seed=1 ===
Seed 1 -> MAE 0.6149, RMSE 0.8220, R2 0.8485, Spearman 0.9143

=== Training Run with seed=2 ===
Seed 2 -> MAE 0.6163, RMSE 0.8227, R2 0.8484, Spearman 0.9139

=== Training Run with seed=3 ===
Seed 3 -> MAE 0.6175, RMSE 0.8215, R2 0.8486, Spearman 0.9140

=== Training Run with seed=4 ===
S